In [1]:
import sys
import math
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score
from collections import defaultdict

import torch
from torch import nn

class Struct:
    def __init__(self, **entries):
        self.__dict__.update(entries)

def adjust_learning_rate(n_epoch_warmup, n_epoch, max_lr, optimizer, dloader, step):
    """
    Set learning rate according to cosine schedule
    """

    max_steps = int(n_epoch * len(dloader))
    warmup_steps = int(n_epoch_warmup * len(dloader))
    
    if step < warmup_steps:
        lr = max_lr * step / warmup_steps
    else:
        step -= warmup_steps
        max_steps -= warmup_steps
        q = 0.5 * (1 + math.cos(math.pi * step / max_steps))
        end_lr = max_lr * 0.001
        lr = max_lr * q + end_lr * (1 - q)

    optimizer.param_groups[0]['lr'] = lr

def shuffle_batch(x, shuffle_idx=None):
    """ shuffles each instance in batch the same way """
    
    if not torch.is_tensor(shuffle_idx):
        seq_len = x.shape[1]
        shuffle_idx = torch.randperm(seq_len)
    x = x[:, shuffle_idx]
    
    return x, shuffle_idx

def shuffle_instance(x, axis, shuffle_idx=None):
    """ shuffles each instance in batch in a different way """

    if not torch.is_tensor(shuffle_idx):
        # get permutation indices
        shuffle_idx = torch.rand(x.shape[:axis+1], device=x.device).argsort(axis)  
    
    idx_expand = shuffle_idx.clone().to(x.device)
    for _ in range(x.ndim-axis-1):
        idx_expand.unsqueeze_(-1)
    # reformat for gather operation
    idx_expand = idx_expand.repeat(*[1 for _ in range(axis+1)], *(x.shape[axis+1:]))  
    
    x = x.gather(axis, idx_expand)

    return x, shuffle_idx

class Logger(nn.Module):
    ''' Stores and computes statistiscs of losses and metrics '''

    def __init__(self, task_dict):
        super().__init__()

        self.task_dict = task_dict
        self.losses_it = defaultdict(list)
        self.losses_epoch = defaultdict(list)
        self.y_preds = defaultdict(list)
        self.y_trues = defaultdict(list)
        self.metrics = defaultdict(list)

    def update(self, next_loss, next_y_pred, next_y_true):
    
        for task in self.task_dict.values():
            t      = task['name']
            t_metr = task['metric']
            # Normalise : metric peut être str ou list
            if isinstance(t_metr, str):
                t_metr = [t_metr]
    
            self.losses_it[t].append(next_loss[t])
    
            # Décider comment stocker y_pred selon les métriques demandées
            has_auc_f1 = any(m in ['auc', 'f1', 'precision', 'recall']
                             for m in t_metr)
    
            if t_metr == ['accuracy']:
                # Seule accuracy → argmax suffit
                y_pred = np.argmax(next_y_pred[t], axis=-1)
            elif has_auc_f1:
                # AUC/F1/precision/recall ont besoin des scores bruts
                raw = next_y_pred[t]
                if raw.ndim == 2:
                    # softmax output [B, n_class] → garder proba classe positive (index 1)
                    y_pred = raw[:, 1].tolist()
                else:
                    # sigmoid output [B] → déjà un scalaire par sample
                    y_pred = raw.tolist()
            else:
                y_pred = next_y_pred[t].tolist()
    
            self.y_preds[t].extend(y_pred)
            self.y_trues[t].extend(next_y_true[t])

    def compute_metric(self):
    
        for task in self.task_dict.values():
            t = task['name']
            losses = self.losses_it[t]
            self.losses_epoch[t].append(np.mean(losses))
    
            # Collect all requested metrics (list or single string)
            requested = task['metric']
            if isinstance(requested, str):
                requested = [requested]
    
            y_true     = np.array(self.y_trues[t])
            y_pred_raw = np.array(self.y_preds[t])
    
            epoch_metrics = {}
            for current_metric in requested:
    
                if current_metric == 'accuracy':
                    # y_pred_raw peut être vecteurs [B, n_class] ou scalaires [B]
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        # scalaires = proba classe 1 → seuil 0.5
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    epoch_metrics[current_metric] = accuracy_score(
                        y_true, y_pred)
    
                elif current_metric == 'multilabel_accuracy':
                    y_pred = np.where(y_pred_raw >= 0.5, 1., 0.)
                    correct = np.all(y_pred == y_true, axis=-1).sum()
                    epoch_metrics[current_metric] = correct / len(y_true)
    
                elif current_metric == 'auc':
                    # robuste aux deux formats
                    scores = y_pred_raw[:, 1] \
                             if y_pred_raw.ndim == 2 else y_pred_raw
                    epoch_metrics[current_metric] = roc_auc_score(
                        y_true, scores)
    
                elif current_metric == 'f1':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    from sklearn.metrics import f1_score
                    epoch_metrics[current_metric] = f1_score(
                        y_true, y_pred, zero_division=0)
    
                elif current_metric == 'precision':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    from sklearn.metrics import precision_score
                    epoch_metrics[current_metric] = precision_score(
                        y_true, y_pred, zero_division=0)
    
                elif current_metric == 'recall':
                    if y_pred_raw.ndim == 2:
                        y_pred = np.argmax(y_pred_raw, axis=-1)
                    else:
                        y_pred = (y_pred_raw >= 0.5).astype(int)
                    from sklearn.metrics import recall_score
                    epoch_metrics[current_metric] = recall_score(
                        y_true, y_pred, zero_division=0)
    
            self.metrics[t].append(epoch_metrics)
    
            # reset
            self.losses_it[t] = []
            self.y_preds[t]   = []
            self.y_trues[t]   = []
    
    def print_stats(self, epoch, train, **kwargs):
    
        print_str  = 'Train' if train else 'Test'
        print_str += " Epoch: {}\n".format(epoch + 1)
    
        avg_loss = 0
        for task in self.task_dict.values():
            t         = task['name']
            mean_loss = self.losses_epoch[t][epoch]
            avg_loss += mean_loss
    
            epoch_metrics = self.metrics[t][epoch]
            metrics_str   = ', '.join(
                f"{k}: {v:.5f}" for k, v in epoch_metrics.items()
            )
            print_str += "task: {}, loss: {:.5f}, {}\n".format(
                t, mean_loss, metrics_str)
    
        avg_loss /= len(self.task_dict.values())
        print_str += "avg loss: {:.5f}".format(avg_loss)
    
        for k, v in kwargs.items():
            print_str += ", {}: {}".format(k, v)
        print_str += "\n"
    
        print(print_str)

In [2]:
# ============================================================
# MS-IPS : Multi-Scale IPS with EfficientNet-B0
# Nouveau notebook — séparé de l'IPS original
# ============================================================

# Guards
run_node21_2d = True    # MS-IPS-2D sur NODE21
run_luna_3d   = False   # MS-IPS-3D sur LUNA16 (plus tard)

# Vérifier que efficientnet est disponible
import torch
import torchvision
print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Vérifier EfficientNet disponible nativement dans torchvision >= 0.13
try:
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    print("EfficientNet-B0 : disponible ✓")
except ImportError:
    print("EfficientNet-B0 : non disponible → pip install torchvision --upgrade")

# Compter les paramètres
import torch.nn as nn
from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights

def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

eff_b0  = efficientnet_b0(weights=None)
res18   = resnet18(weights=None)
res50   = resnet50(weights=None)

print(f"\n=== Comparaison encodeurs ===")
print(f"ResNet18        : {count_params(res18):.1f}M params")
print(f"ResNet50        : {count_params(res50):.1f}M params")
print(f"EfficientNet-B0 : {count_params(eff_b0):.1f}M params")

# Output dim de chaque encodeur
# ResNet18 → 512, ResNet50 → 2048, EfficientNet-B0 → 1280
print(f"\n=== Output dimensions ===")
dummy = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    out18  = resnet18(weights=None)
    out18.fc = nn.Identity()
    out50  = resnet50(weights=None)
    out50.fc = nn.Identity()
    eff    = efficientnet_b0(weights=None)
    eff.classifier = nn.Identity()
    print(f"ResNet18        : {out18(dummy).shape}")
    print(f"ResNet50        : {out50(dummy).shape}")
    print(f"EfficientNet-B0 : {eff(dummy).shape}")

PyTorch      : 2.10.0+cu128
Torchvision  : 0.25.0+cu128
CUDA         : True
GPU          : Tesla T4
VRAM total   : 15.6 GB
EfficientNet-B0 : disponible ✓

=== Comparaison encodeurs ===
ResNet18        : 11.7M params
ResNet50        : 25.6M params
EfficientNet-B0 : 5.3M params

=== Output dimensions ===
ResNet18        : torch.Size([1, 512])
ResNet50        : torch.Size([1, 2048])
EfficientNet-B0 : torch.Size([1, 1280])


## Cellule 1 — Encodeur EfficientNet-B0 adapté à 1 canal

In [3]:
# ============================================================
# Section 1 — Encodeur EfficientNet-B0 patch encoder
# Adapté pour 1 canal (grayscale CT/CXR) ou 3 canaux (RGB)
# Output : feature vector D=1280 par patch
# ============================================================
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class EfficientNetPatchEncoder(nn.Module):
    """
    EfficientNet-B0 adapté comme encodeur de patches pour MS-IPS.

    Différences vs IPS original (ResNet) :
    - 5.3M params vs 11.7M (ResNet18) ou 25.6M (ResNet50)
    - Output dim 1280 vs 512 / 2048
    - Support natif 1 ou 3 canaux d'entrée
    - Compound scaling → meilleur accuracy/FLOP ratio

    Args:
        n_chan_in : nombre de canaux d'entrée (1 pour CT/CXR, 3 pour RGB)
        pretrained: utiliser les poids ImageNet
        freeze_bn : geler les BatchNorm pendant IPS (no-grad mode)
    """

    def __init__(self, n_chan_in=1, pretrained=True, freeze_bn=False):
        super().__init__()

        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        base    = efficientnet_b0(weights=weights)

        # Adapter le premier conv au bon nombre de canaux
        # EfficientNet-B0 stem : Conv2d(3, 32, kernel_size=3, stride=2, padding=1)
        if n_chan_in != 3:
            old_conv = base.features[0][0]
            base.features[0][0] = nn.Conv2d(
                n_chan_in,
                old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=old_conv.bias is not None
            )
            # Si pretrained, initialiser en moyennant sur les canaux RGB
            if pretrained and n_chan_in == 1:
                with torch.no_grad():
                    base.features[0][0].weight.copy_(
                        old_conv.weight.mean(dim=1, keepdim=True)
                    )

        # Garder features + avgpool, supprimer classifier
        self.features = base.features
        self.avgpool  = base.avgpool
        # Output dim fixe = 1280
        self.output_dim = 1280

        self.freeze_bn = freeze_bn

    def forward(self, x):
        """
        x : (B, n_chan_in, H, W)
        returns : (B, 1280)
        """
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return x

    def train(self, mode=True):
        super().train(mode)
        if self.freeze_bn and mode:
            # Geler les BN pendant l'entraînement (utile pour petits batchs)
            for m in self.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                    m.eval()
        return self


# ── Tests ──────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# Test 1 canal (CXR/CT)
enc_1ch = EfficientNetPatchEncoder(n_chan_in=1, pretrained=True).to(device)
x1 = torch.zeros(8, 1, 128, 128).to(device)   # batch de 8 patches 128×128
with torch.no_grad():
    out1 = enc_1ch(x1)
print(f"\n1 canal  — input {tuple(x1.shape)} → output {tuple(out1.shape)}")

# Test 3 canaux (RGB)
enc_3ch = EfficientNetPatchEncoder(n_chan_in=3, pretrained=True).to(device)
x3 = torch.zeros(8, 3, 128, 128).to(device)
with torch.no_grad():
    out3 = enc_3ch(x3)
print(f"3 canaux — input {tuple(x3.shape)} → output {tuple(out3.shape)}")

# Vérifier output dim
assert out1.shape == (8, 1280), f"Expected (8, 1280), got {out1.shape}"
assert out3.shape == (8, 1280), f"Expected (8, 1280), got {out3.shape}"
print("✓ Output dim correct : 1280")

# Compter les paramètres
n_params = sum(p.numel() for p in enc_1ch.parameters()) / 1e6
print(f"✓ Paramètres : {n_params:.1f}M")

# VRAM après chargement
torch.cuda.empty_cache()
vram_used = torch.cuda.memory_allocated() / 1e9
print(f"✓ VRAM utilisée après chargement : {vram_used:.3f} GB")

# Test avec N patches en parallèle (simulation IPS no-grad)
N = 64   # 64 patches par image (1024×1024 avec stride 128)
x_all = torch.zeros(N, 1, 128, 128).to(device)
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    feats_all = enc_1ch(x_all)
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"\n=== Simulation IPS no-grad ({N} patches) ===")
print(f"Peak VRAM : {peak:.3f} GB")
print(f"Features  : {tuple(feats_all.shape)}")

Device : cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]



1 canal  — input (8, 1, 128, 128) → output (8, 1280)
3 canaux — input (8, 3, 128, 128) → output (8, 1280)
✓ Output dim correct : 1280
✓ Paramètres : 4.0M
✓ VRAM utilisée après chargement : 0.035 GB

=== Simulation IPS no-grad (64 patches) ===
Peak VRAM : 0.257 GB
Features  : (64, 1280)


## Cellule 2 — Patchifier multi-échelle

In [4]:
class MultiScalePatchifier(nn.Module):
    """
    Découpe une image en patches à 2 échelles.

    Design corrigé pour NODE21 (1024×1024) :
      Grossier : 256×256, stride=256 → 16 patches (contexte)
      Fin      : 128×128, stride=128 → 64 patches (détail)
      Total N  : 80 patches — comparable à IPS original (64)

    Les deux tailles de patches sont passées à l'encodeur
    telles quelles — l'encodeur EfficientNet-B0 accepte
    n'importe quelle résolution ≥ 32×32.
    On ne les redimensionne PAS à la même taille :
    chaque patch passe dans l'encodeur à sa résolution native.
    L'output est toujours (B, 1280) grâce à l'avgpool global.
    """

    def __init__(self,
                 coarse_size=256,
                 coarse_stride=256,
                 fine_size=128,
                 fine_stride=128,
                 img_size=1024,
                 n_chan=1):
        super().__init__()
        self.coarse_size   = coarse_size
        self.coarse_stride = coarse_stride
        self.fine_size     = fine_size
        self.fine_stride   = fine_stride
        self.img_size      = img_size
        self.n_chan        = n_chan

        self.n_coarse = ((img_size - coarse_size) // coarse_stride + 1) ** 2
        self.n_fine   = ((img_size - fine_size)   // fine_stride   + 1) ** 2
        self.n_total  = self.n_coarse + self.n_fine

        print(f"MultiScalePatchifier :")
        print(f"  Grossier : {coarse_size}×{coarse_size}, "
              f"stride={coarse_stride} → {self.n_coarse} patches")
        print(f"  Fin      : {fine_size}×{fine_size}, "
              f"stride={fine_stride} → {self.n_fine} patches")
        print(f"  Total N  : {self.n_total} patches")

    def extract_patches(self, img, patch_size, stride):
        """img : (C, H, W) → (N, C, patch_size, patch_size)"""
        patches = img.unfold(1, patch_size, stride) \
                     .unfold(2, patch_size, stride)
        patches = patches.permute(1, 2, 0, 3, 4)
        patches = patches.reshape(-1, self.n_chan, patch_size, patch_size)
        return patches

    def forward(self, img):
        """
        img : (C, H, W)
        Returns:
            p_coarse : (N_c, C, coarse_size, coarse_size)
            p_fine   : (N_f, C, fine_size,   fine_size)
            — séparés car tailles différentes
        """
        p_coarse = self.extract_patches(
            img, self.coarse_size, self.coarse_stride)
        p_fine = self.extract_patches(
            img, self.fine_size, self.fine_stride)
        return p_coarse, p_fine


# ── Test ───────────────────────────────────────────────────
patchifier = MultiScalePatchifier(
    coarse_size=256, coarse_stride=256,
    fine_size=128,   fine_stride=128,
    img_size=1024,   n_chan=1
)

img = torch.zeros(1, 1024, 1024)
p_c, p_f = patchifier(img)
print(f"\nGrossier : {tuple(p_c.shape)}")
print(f"Fin      : {tuple(p_f.shape)}")

# Encoder les deux échelles séparément — tailles natives
enc = EfficientNetPatchEncoder(n_chan_in=1, pretrained=False).to(device)

torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    f_c = enc(p_c.to(device))   # (16, 1280)
    f_f = enc(p_f.to(device))   # (64, 1280)
peak = torch.cuda.max_memory_allocated() / 1e9

# Flag d'échelle
scale_c = torch.zeros(len(p_c), dtype=torch.long)   # 0 = grossier
scale_f = torch.ones(len(p_f),  dtype=torch.long)   # 1 = fin

# Concaténer features + flags
feats_all  = torch.cat([f_c, f_f], dim=0)            # (80, 1280)
scales_all = torch.cat([scale_c, scale_f], dim=0)    # (80,)

print(f"\n=== Features combinées ===")
print(f"feats_all  : {tuple(feats_all.shape)}")
print(f"scales_all : {tuple(scales_all.shape)}")
print(f"Peak VRAM (no-grad, N=80) : {peak:.3f} GB")

# Comparer correctement avec IPS original
from torchvision.models import resnet18
res18 = resnet18(weights=None)
res18.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
res18.fc    = nn.Identity()
res18       = res18.to(device)

# IPS original : 64 patches 128×128
x_ips = torch.zeros(64, 1, 128, 128).to(device)
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    _ = res18(x_ips)
peak_ips = torch.cuda.max_memory_allocated() / 1e9

print(f"\n=== Comparaison VRAM (no-grad) ===")
print(f"MS-IPS EfficientNet-B0 (80 patches, 2 échelles) : {peak:.3f} GB")
print(f"IPS    ResNet18         (64 patches, 1 échelle)  : {peak_ips:.3f} GB")
print(f"Overhead multi-échelle : {(peak/peak_ips - 1)*100:.0f}%")
print(f"\n=== Bilan comparatif ===")
print(f"{'':30s} {'MS-IPS':>10} {'IPS':>10}")
print(f"{'Params encodeur':30s} {'4.0M':>10} {'11.7M':>10}")
print(f"{'N patches':30s} {'80':>10} {'64':>10}")
print(f"{'VRAM no-grad':30s} {peak:>9.3f}G {peak_ips:>9.3f}G")

MultiScalePatchifier :
  Grossier : 256×256, stride=256 → 16 patches
  Fin      : 128×128, stride=128 → 64 patches
  Total N  : 80 patches

Grossier : (16, 1, 256, 256)
Fin      : (64, 1, 128, 128)

=== Features combinées ===
feats_all  : (80, 1280)
scales_all : (80,)
Peak VRAM (no-grad, N=80) : 0.279 GB

=== Comparaison VRAM (no-grad) ===
MS-IPS EfficientNet-B0 (80 patches, 2 échelles) : 0.279 GB
IPS    ResNet18         (64 patches, 1 échelle)  : 0.249 GB
Overhead multi-échelle : 12%

=== Bilan comparatif ===
                                   MS-IPS        IPS
Params encodeur                      4.0M      11.7M
N patches                              80         64
VRAM no-grad                       0.279G     0.249G


## Cellule 3 — Transformer multi-échelle (scorer + agrégateur)

In [5]:
# ============================================================
# Section 3 — MS-Transformer
# Identique au transformer IPS original +
# embedding d'échelle injecté dans les features
# ============================================================
import math
import torch
import torch.nn as nn

class ScaledDotProductAttention(nn.Module):
    """Identique à IPS original."""
    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout     = nn.Dropout(attn_dropout)

    def compute_attn(self, q, k):
        attn = torch.matmul(q / self.temperature, k.transpose(2, 3))
        attn = self.dropout(torch.softmax(attn, dim=-1))
        return attn

    def forward(self, q, k, v):
        attn   = self.compute_attn(q, k)
        output = torch.matmul(attn, v)
        return output


class MultiHeadCrossAttention(nn.Module):
    """Identique à IPS original."""
    def __init__(self, n_token, H, D, D_k, D_v,
                 attn_dropout=0.1, dropout=0.1):
        super().__init__()
        self.n_token = n_token
        self.H       = H
        self.D_k     = D_k
        self.D_v     = D_v

        self.q = nn.Parameter(torch.empty((1, n_token, D)))
        nn.init.uniform_(self.q, a=-math.sqrt(1/D_k),
                                  b= math.sqrt(1/D_k))

        self.q_w = nn.Linear(D, H * D_k, bias=False)
        self.k_w = nn.Linear(D, H * D_k, bias=False)
        self.v_w = nn.Linear(D, H * D_v, bias=False)
        self.fc  = nn.Linear(H * D_v, D, bias=False)

        self.attention  = ScaledDotProductAttention(
            temperature=D_k ** 0.5, attn_dropout=attn_dropout)
        self.dropout    = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)

    def get_attn(self, x):
        D_k, H, n_token = self.D_k, self.H, self.n_token
        B, len_seq      = x.shape[:2]
        q = self.q_w(self.q).view(1, n_token, H, D_k)
        k = self.k_w(x).view(B, len_seq, H, D_k)
        q, k = q.transpose(1, 2), k.transpose(1, 2)
        return self.attention.compute_attn(q, k)

    def forward(self, x):
        D_k, D_v, H, n_token = self.D_k, self.D_v, self.H, self.n_token
        B, len_seq = x.shape[:2]
        q = self.q_w(self.q).view(1, n_token, H, D_k)
        k = self.k_w(x).view(B, len_seq, H, D_k)
        v = self.v_w(x).view(B, len_seq, H, D_v)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        x = self.attention(q, k, v)
        x = x.transpose(1, 2).contiguous().view(B, n_token, -1)
        x = self.dropout(self.fc(x))
        x += self.q
        x = self.layer_norm(x)
        return x


class MLP(nn.Module):
    """Identique à IPS original."""
    def __init__(self, D, D_inner, dropout=0.1):
        super().__init__()
        self.w_1        = nn.Linear(D, D_inner)
        self.w_2        = nn.Linear(D_inner, D)
        self.layer_norm = nn.LayerNorm(D, eps=1e-6)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.w_2(torch.relu(self.w_1(x)))
        x = self.dropout(x)
        x += residual
        x = self.layer_norm(x)
        return x


class MSTransformer(nn.Module):
    """
    Cross-attention transformer pour MS-IPS.

    Différence vs IPS original :
    - Prend en entrée les features enrichies avec le
      scale embedding (déjà ajouté avant d'arriver ici)
    - Interface identique : get_scores() + forward()
    """
    def __init__(self, n_token, H, D, D_k, D_v,
                 D_inner, attn_dropout=0.1, dropout=0.1):
        super().__init__()
        self.crs_attn = MultiHeadCrossAttention(
            n_token, H, D, D_k, D_v,
            attn_dropout=attn_dropout, dropout=dropout)
        self.mlp = MLP(D, D_inner, dropout=dropout)

    def get_scores(self, x):
        attn = self.crs_attn.get_attn(x)
        # Moyenne sur têtes et tokens — identique à IPS
        return attn.mean(dim=1).transpose(1, 2).mean(-1)

    def forward(self, x):
        return self.mlp(self.crs_attn(x))


class ScaleEmbedding(nn.Module):
    """
    Injecte l'information d'échelle dans les features.

    Ajoute un vecteur appris de dimension D à chaque
    feature selon son échelle (0=grossier, 1=fin).
    Permet au transformer de distinguer les deux échelles
    sans modifier l'architecture du cross-attention.

    C'est la seule vraie nouveauté architecturale
    par rapport à IPS original.
    """
    def __init__(self, D, n_scales=2):
        super().__init__()
        # Table d'embeddings : 2 vecteurs de dim D
        self.embedding = nn.Embedding(n_scales, D)
        nn.init.normal_(self.embedding.weight, std=0.02)

    def forward(self, features, scales):
        """
        features : (B, N, D) ou (N, D)
        scales   : (N,) — 0 ou 1
        returns  : features + scale_embedding, même shape
        """
        scale_emb = self.embedding(scales.to(features.device))
        # scale_emb : (N, D)
        if features.dim() == 3:
            # batch mode : (B, N, D)
            return features + scale_emb.unsqueeze(0)
        else:
            # single mode : (N, D)
            return features + scale_emb


# ── Tests ──────────────────────────────────────────────────
D      = 512
D_k    = 64
D_v    = 64
D_inner= 2048
H      = 8
n_token= 1

# Projector : 1280 → D (comme IPS original pour Camelyon)
projector = nn.Sequential(
    nn.LayerNorm(1280, elementwise_affine=False),
    nn.Linear(1280, D),
    nn.BatchNorm1d(D),
    nn.ReLU()
).to(device)

scale_emb   = ScaleEmbedding(D).to(device)
ms_transf   = MSTransformer(n_token, H, D, D_k, D_v,
                             D_inner).to(device)

# Simulation : batch de 1 image, 80 patches (16 grossiers + 64 fins)
N   = 80
B   = 1
feats_raw = torch.randn(N, 1280).to(device)      # features brutes
scales    = torch.cat([torch.zeros(16, dtype=torch.long),
                       torch.ones(64,  dtype=torch.long)])

# Pipeline complet
with torch.no_grad():
    # 1. Projeter 1280 → D
    feats_proj = projector(feats_raw)              # (N, D)
    # 2. Ajouter scale embedding
    feats_scaled = scale_emb(feats_proj, scales)   # (N, D)
    # 3. Scorer pour IPS (format batch : (1, N, D))
    feats_batch = feats_scaled.unsqueeze(0)         # (1, N, D)
    scores = ms_transf.get_scores(feats_batch)      # (1, N)
    # 4. Agréger après sélection (simulation M=20)
    M        = 20
    top_idx  = torch.topk(scores, M, dim=-1)[1]    # (1, M)
    mem_emb  = torch.gather(feats_batch, 1,
                  top_idx.unsqueeze(-1).expand(-1, -1, D))  # (1, M, D)
    out      = ms_transf(mem_emb)                  # (1, n_token, D)

print("=== Test MS-Transformer ===")
print(f"Input features  : (N={N}, D_raw=1280)")
print(f"After projection: (N={N}, D={D})")
print(f"After scale emb : (N={N}, D={D})")
print(f"Scores shape    : {tuple(scores.shape)}")
print(f"Top-{M} selected : {tuple(mem_emb.shape)}")
print(f"Transformer out : {tuple(out.shape)}")

# Vérifier que les scores sont différents
# entre patches grossiers et fins
scores_np = scores.squeeze().cpu()
print(f"\n=== Scores par échelle ===")
print(f"Grossier (idx 0-15)  — "
      f"mean={scores_np[:16].mean():.4f}, "
      f"std={scores_np[:16].std():.4f}")
print(f"Fin      (idx 16-79) — "
      f"mean={scores_np[16:].mean():.4f}, "
      f"std={scores_np[16:].std():.4f}")

# Compter combien de patches de chaque échelle
# sont dans le Top-M
top_idx_np = top_idx.squeeze().cpu()
n_coarse_selected = (top_idx_np < 16).sum().item()
n_fine_selected   = (top_idx_np >= 16).sum().item()
print(f"\nDans Top-{M} sélectionnés :")
print(f"  Grossiers : {n_coarse_selected}/{16}")
print(f"  Fins      : {n_fine_selected}/{64}")

# Paramètres totaux MS-IPS vs IPS
n_proj    = sum(p.numel() for p in projector.parameters())
n_semb    = sum(p.numel() for p in scale_emb.parameters())
n_transf  = sum(p.numel() for p in ms_transf.parameters())
n_enc     = 4_000_000  # EfficientNet-B0 4M

n_msips  = (n_enc + n_proj + n_semb + n_transf) / 1e6
n_ips    = (11_700_000 + n_transf) / 1e6  # ResNet18 + même transformer

print(f"\n=== Paramètres totaux ===")
print(f"MS-IPS : encodeur {n_enc/1e6:.1f}M + "
      f"projector {n_proj/1e6:.2f}M + "
      f"scale_emb {n_semb/1e6:.4f}M + "
      f"transformer {n_transf/1e6:.2f}M "
      f"= {n_msips:.2f}M total")
print(f"IPS    : ResNet18 11.7M + "
      f"transformer {n_transf/1e6:.2f}M "
      f"= {n_ips:.2f}M total")
print(f"Réduction : {(1 - n_msips/n_ips)*100:.0f}%")

=== Test MS-Transformer ===
Input features  : (N=80, D_raw=1280)
After projection: (N=80, D=512)
After scale emb : (N=80, D=512)
Scores shape    : (1, 80)
Top-20 selected : (1, 20, 512)
Transformer out : (1, 1, 512)

=== Scores par échelle ===
Grossier (idx 0-15)  — mean=0.0129, std=0.0012
Fin      (idx 16-79) — mean=0.0126, std=0.0014

Dans Top-20 sélectionnés :
  Grossiers : 6/16
  Fins      : 14/64

=== Paramètres totaux ===
MS-IPS : encodeur 4.0M + projector 0.66M + scale_emb 0.0010M + transformer 3.15M = 7.81M total
IPS    : ResNet18 11.7M + transformer 3.15M = 14.85M total
Réduction : 47%


## Cellule 4 — MS-IPS Net complet

In [6]:
# ============================================================
# Section 4 — MS-IPSNet : modèle complet
# Intègre EfficientNet-B0 + MultiScalePatchifier +
# ScaleEmbedding + MSTransformer + ClassHead
# Architecture parallèle à IPSNet original
# ============================================================
import math
import torch
import torch.nn as nn

class MSIPSNet(nn.Module):
    """
    Multi-Scale IPS Network.

    Pipeline :
    1. Patchifier : image → patches grossiers + fins
    2. IPS (no-grad) : sélection Top-M parmi N_c + N_f patches
       - encode les deux échelles avec EfficientNet-B0
       - projette 1280 → D
       - ajoute scale embedding
       - score iteratif, garde Top-M
    3. Forward (grad) : re-encode + agrège + classifie
       - re-encode les M patches sélectionnés
       - projette + scale embedding
       - MSTransformer cross-attention
       - tête de classification

    Args:
        device     : torch device
        conf       : Struct avec les hyperparamètres
    """

    def __init__(self, device, conf):
        super().__init__()
        self.device       = device
        self.M            = conf.M
        self.I            = conf.I
        self.D            = conf.D
        self.tasks        = conf.tasks
        self.n_class      = conf.n_class
        self.shuffle      = conf.shuffle
        self.shuffle_style= conf.shuffle_style

        # ── Patchifier multi-échelle ─────────────────────────
        self.patchifier = MultiScalePatchifier(
            coarse_size=conf.coarse_size,
            coarse_stride=conf.coarse_stride,
            fine_size=conf.fine_size,
            fine_stride=conf.fine_stride,
            img_size=conf.img_size,
            n_chan=conf.n_chan_in
        )

        # ── Encodeur EfficientNet-B0 ─────────────────────────
        self.encoder = EfficientNetPatchEncoder(
            n_chan_in=conf.n_chan_in,
            pretrained=conf.pretrained
        )
        enc_out_dim = 1280

        # ── Projector 1280 → D ──────────────────────────────
        self.projector = nn.Sequential(
            nn.LayerNorm(enc_out_dim, elementwise_affine=False),
            nn.Linear(enc_out_dim, conf.D),
            nn.BatchNorm1d(conf.D),
            nn.ReLU()
        )

        # ── Scale embedding ──────────────────────────────────
        self.scale_emb = ScaleEmbedding(conf.D, n_scales=2)

        # ── Transformer ──────────────────────────────────────
        self.transf = MSTransformer(
            conf.n_token, conf.H, conf.D,
            conf.D_k, conf.D_v, conf.D_inner,
            conf.attn_dropout, conf.dropout
        )

        # ── Têtes de classification ───────────────────────────
        self.output_layers = self._build_output_layers(conf.tasks)

    def _build_output_layers(self, tasks):
        layers = nn.ModuleDict()
        for task in tasks.values():
            if task['act_fn'] == 'softmax':
                act = nn.Softmax(dim=-1)
            elif task['act_fn'] == 'sigmoid':
                act = nn.Sigmoid()
            layers[task['name']] = nn.Sequential(
                nn.Linear(self.D, self.n_class),
                act
            )
        return layers

    def _encode_and_project(self, patches, scales):
        """
        patches : (N, C, H, W)
        scales  : (N,) — 0 ou 1
        returns : (N, D) features projetées + scale embedding
        """
        N = patches.shape[0]
        # Encoder par batch pour éviter OOM
        BATCH = 128
        feats = []
        for i in range(0, N, BATCH):
            f = self.encoder(patches[i:i+BATCH])
            feats.append(f)
        feats = torch.cat(feats, dim=0)        # (N, 1280)
        feats = self.projector(feats)          # (N, D)
        feats = self.scale_emb(feats, scales)  # (N, D)
        return feats

    def _score_and_select(self, emb, M, idx):
        """
        emb : (B, N, D)
        idx : (B, N)
        returns : mem_emb (B, M, D), mem_idx (B, M)
        """
        attn    = self.transf.get_scores(emb)         # (B, N)
        top_idx = torch.topk(attn, M, dim=-1)[1]      # (B, M)
        mem_emb = torch.gather(emb, 1,
            top_idx.unsqueeze(-1).expand(-1, -1, self.D))
        mem_idx = torch.gather(idx, 1, top_idx)
        return mem_emb, mem_idx

    @torch.no_grad()
    def ips(self, img):
        """
        Iterative Patch Selection multi-échelle.

        img : (C, H, W) — une image (B_seq=1)
        returns :
            mem_patches_c : (M_c, C, coarse_size, coarse_size)
            mem_patches_f : (M_f, C, fine_size, fine_size)
            mem_scales    : (M,) — échelle de chaque patch sélectionné
            mem_idx       : (M,)
        """
        M, I, D = self.M, self.I, self.D
        device  = self.device

        # IPS en eval mode
        if self.training:
            self.encoder.eval()
            self.transf.eval()
            self.projector.eval()

        # 1. Patchifier
        p_coarse, p_fine = self.patchifier(img)
        # p_coarse : (N_c, C, coarse_size, coarse_size)
        # p_fine   : (N_f, C, fine_size, fine_size)
        N_c = len(p_coarse)
        N_f = len(p_fine)
        N   = N_c + N_f

        # Concaténer dans une liste indexée
        # Index 0..N_c-1 = grossiers, N_c..N-1 = fins
        scales_all = torch.cat([
            torch.zeros(N_c, dtype=torch.long),
            torch.ones(N_f,  dtype=torch.long)
        ])  # (N,)

        # Shortcut : si M >= N, pas besoin d'IPS
        if M >= N:
            # Redimensionner les fins à coarse_size pour uniformité
            import torch.nn.functional as F
            p_fine_up = F.interpolate(
                p_fine.to(device),
                size=(self.patchifier.coarse_size,
                      self.patchifier.coarse_size),
                mode='bilinear', align_corners=False)
            all_patches = torch.cat([p_coarse.to(device),
                                     p_fine_up], dim=0)
            if self.training:
                self.encoder.train()
                self.transf.train()
                self.projector.train()
            return all_patches, scales_all.to(device), \
                   torch.arange(N, device=device)

        # 2. Encoder par échelle (tailles natives)
        p_coarse_dev = p_coarse.to(device)
        p_fine_dev   = p_fine.to(device)

        # Encoder grossiers
        f_c = self.encoder(
            p_coarse_dev.reshape(-1, *p_coarse_dev.shape[1:]))
        f_c = f_c.view(N_c, -1)                       # (N_c, 1280)

        # Encoder fins
        f_f = self.encoder(
            p_fine_dev.reshape(-1, *p_fine_dev.shape[1:]))
        f_f = f_f.view(N_f, -1)                       # (N_f, 1280)

        # Concaténer et projeter
        feats_all = torch.cat([f_c, f_f], dim=0)      # (N, 1280)
        feats_all = self.projector(feats_all)          # (N, D)
        feats_all = self.scale_emb(
            feats_all, scales_all.to(device))          # (N, D)

        # 3. IPS itératif
        idx_all = torch.arange(N, device=device).unsqueeze(0)  # (1, N)

        # Init buffer mémoire avec les M premiers patches
        mem_emb = feats_all[:M].unsqueeze(0)          # (1, M, D)
        mem_idx = idx_all[:, :M]                      # (1, M)

        n_iter = math.ceil((N - M) / I)
        for i in range(n_iter):
            start = i * I + M
            end   = min(start + I, N)
            iter_emb = feats_all[start:end].unsqueeze(0)  # (1, I, D)
            iter_idx = idx_all[:, start:end]              # (1, I)
            all_emb  = torch.cat([mem_emb, iter_emb], dim=1)
            all_idx  = torch.cat([mem_idx, iter_idx], dim=1)
            mem_emb, mem_idx = self._score_and_select(
                all_emb, M, all_idx)

        # 4. Récupérer les patches sélectionnés
        mem_idx_flat = mem_idx.squeeze(0)              # (M,)
        mem_scales   = scales_all[mem_idx_flat.cpu()]  # (M,)

        # Reconstruire les patches sélectionnés
        # Indices < N_c → patch grossier, sinon fin
        import torch.nn.functional as F
        selected_patches = []
        for idx_val in mem_idx_flat.cpu().tolist():
            if idx_val < N_c:
                patch = p_coarse[idx_val]              # (C, coarse, coarse)
            else:
                patch = p_fine[idx_val - N_c]          # (C, fine, fine)
                # Redimensionner à coarse_size pour uniformité
                patch = F.interpolate(
                    patch.unsqueeze(0),
                    size=(self.patchifier.coarse_size,
                          self.patchifier.coarse_size),
                    mode='bilinear', align_corners=False
                ).squeeze(0)
            selected_patches.append(patch)

        mem_patches = torch.stack(selected_patches).to(device)  # (M, C, H, W)

        if self.training:
            self.encoder.train()
            self.transf.train()
            self.projector.train()

        return mem_patches, mem_scales.to(device), mem_idx_flat.to(device)

    def forward(self, mem_patches, mem_scales):
        M = mem_patches.shape[0]
    
        # Re-encode avec gradient checkpointing pour économiser VRAM
        # Recompute les activations pendant la backprop au lieu de les stocker
        from torch.utils.checkpoint import checkpoint
    
        def encode_fn(patches):
            return self.encoder(patches)
    
        feats = checkpoint(encode_fn, mem_patches,
                           use_reentrant=False)   # (M, 1280)
        feats = self.projector(feats)              # (M, D)
        feats = self.scale_emb(feats, mem_scales)  # (M, D)
    
        feats = feats.unsqueeze(0)                 # (1, M, D)
        image_emb = self.transf(feats)             # (1, n_token, D)
    
        preds = {}
        for task in self.tasks.values():
            t_name, t_id = task['name'], task['id']
            emb = image_emb[:, t_id]
            preds[t_name] = self.output_layers[t_name](emb)
        return preds

# ── Test rapide ─────────────────────────────────────────────
# Config minimale pour le test
class Conf:
    # patchifier
    coarse_size   = 256
    coarse_stride = 256
    fine_size     = 128
    fine_stride   = 128
    img_size      = 1024
    n_chan_in     = 1
    # encoder
    pretrained    = False
    # ips
    M = 20
    I = 20
    shuffle = False
    shuffle_style = 'batch'
    # transformer
    n_token  = 1
    H        = 4
    D        = 256
    D_k      = 32
    D_v      = 32
    D_inner  = 512
    attn_dropout = 0.1
    dropout      = 0.1
    # task
    n_class = 2
    tasks   = {'task0': {'id': 0, 'name': 'nodule',
                         'act_fn': 'softmax', 'metric': 'accuracy'}}

conf_test = Conf()
net = MSIPSNet(device, conf_test).to(device)
net.eval()

# Image dummy
img = torch.rand(1, 1024, 1024)

print("=== Test MSIPSNet ===")
with torch.no_grad():
    mem_patches, mem_scales, mem_idx = net.ips(img)
    print(f"IPS output :")
    print(f"  mem_patches : {tuple(mem_patches.shape)}")
    print(f"  mem_scales  : {tuple(mem_scales.shape)}")
    print(f"  Grossiers sélectionnés : "
          f"{(mem_scales.cpu()==0).sum().item()}/{conf_test.M}")
    print(f"  Fins sélectionnés      : "
          f"{(mem_scales.cpu()==1).sum().item()}/{conf_test.M}")

    preds = net(mem_patches, mem_scales)
    print(f"\nForward output :")
    for k, v in preds.items():
        print(f"  {k} : {tuple(v.shape)}")

# Paramètres totaux
n_total = sum(p.numel() for p in net.parameters()) / 1e6
print(f"\nParamètres MS-IPS : {n_total:.2f}M")

MultiScalePatchifier :
  Grossier : 256×256, stride=256 → 16 patches
  Fin      : 128×128, stride=128 → 64 patches
  Total N  : 80 patches
=== Test MSIPSNet ===
IPS output :
  mem_patches : (20, 1, 256, 256)
  mem_scales  : (20,)
  Grossiers sélectionnés : 16/20
  Fins sélectionnés      : 4/20

Forward output :
  nodule : (1, 2)

Paramètres MS-IPS : 4.73M


## Cellule 5 — Dataset NODE21 pour MS-IPS

In [7]:
# ============================================================
# Section 5 — NODE21Dataset pour MS-IPS
# Retourne une image entière (C, H, W) — pas des patches
# Le patchifier est dans MSIPSNet.ips()
# ============================================================
import os
import hashlib
import numpy as np
import torch
import SimpleITK as sitk
from pathlib import Path
from torch.utils.data import Dataset
from torchvision import transforms

class NODE21DatasetMSIPS(Dataset):
    """
    Dataset NODE21 pour MS-IPS.

    Différence clé vs NODE21Dataset (IPS original) :
    - Retourne l'IMAGE ENTIÈRE (C, H, W), pas des patches
    - Le découpage en patches multi-échelle est fait
      dans MSIPSNet.ips() à chaque forward pass
    - Normalise en [0,1] puis centre/réduit

    Labels :
      0 = pas de nodule (orig/c*.mha)
      1 = nodule présent (proc/n*.mha)
    """

    IMG_SIZE = 1024

    @staticmethod
    def _to_split(name, seed=42):
        h = int(hashlib.md5(
            (name + str(seed)).encode()).hexdigest(), 16)
        return 'test' if (h % 100) < 20 else 'train'

    def __init__(self, conf, train=True):
        self.tasks  = conf.tasks
        self.train  = train
        self.img_size = conf.img_size

        data_root = Path(conf.data_dir)
        proc_dir  = data_root / 'cxr_images' / 'proccessed_data'
        orig_dir  = data_root / 'cxr_images' / 'original_data'
        
        all_samples = (
            [(f, 1) for f in sorted(proc_dir.rglob('n*.mha'))] +
            [(f, 0) for f in sorted(orig_dir.rglob('c*.mha'))]
        )

        split_name = 'train' if train else 'test'
        self._data = [
            (path, label) for path, label in all_samples
            if self._to_split(path.name) == split_name
        ]

        # Transforms après normalisation [0,1]
        if train:
            self.aug = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(5),
            ])
        else:
            self.aug = None

        pos = sum(l for _, l in self._data)
        neg = len(self._data) - pos
        print(f"{'Train' if train else 'Test'} : "
              f"{len(self._data)} images "
              f"(pos={pos}, neg={neg})")

    def __len__(self):
        return len(self._data)

    def __getitem__(self, i):
        path, label = self._data[i]

        # Charger et normaliser en [0,1]
        arr = sitk.GetArrayFromImage(
            sitk.ReadImage(str(path))).astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)

        # Redimensionner à IMG_SIZE×IMG_SIZE si nécessaire
        if arr.shape != (self.img_size, self.img_size):
            import torch.nn.functional as F
            t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=(self.img_size, self.img_size),
                              mode='bilinear', align_corners=False)
            arr = t.squeeze().numpy()

        # (H, W) → tensor (1, H, W)
        img = torch.from_numpy(arr).unsqueeze(0)

        # Normaliser [-1, 1]
        img = (img - 0.5) / 0.5

        # Augmentations train
        if self.aug is not None:
            img = self.aug(img)

        data_dict = {'input': img}   # (1, H, W)
        for task in self.tasks.values():
            data_dict[task['name']] = label

        return data_dict

## Cellule 6 — Config MS-IPS NODE21

In [8]:
import yaml

msips_node21_yaml = """
#opt
n_epoch: 50
B: 4
B_seq: 1
n_epoch_warmup: 5
lr: 0.0001
wd: 0.1

#dset
n_class: 2
data_dir: '/kaggle/input/datasets/pshikk/node-21-dataset-untampered'
img_size: 1024
n_worker: 2
pin_memory: True
eager: True

#misc
eps: 0.000001
seed: 0
track_efficiency: False
track_epoch: 0

#enc
n_chan_in: 1
pretrained: True

#patchifier multi-échelle — tailles réduites
coarse_size: 128       # ← réduit de 256
coarse_stride: 128
fine_size: 64          # ← réduit de 128
fine_stride: 64

#ips
shuffle: True
shuffle_style: 'batch'
n_token: 1
M: 10                  # ← réduit de 20
I: 10

#aggr
use_pos: False
H: 8
D: 512
D_k: 64
D_v: 64
D_inner: 2048
attn_dropout: 0.1
dropout: 0.1

tasks:
  task0:
    id: 0
    name: 'nodule'
    act_fn: 'softmax'
    metric:
      - accuracy
      - auc
      - f1
      - precision
      - recall
"""

msips_node21_conf = Struct(**yaml.safe_load(msips_node21_yaml))
print("Config MS-IPS NODE21 chargée")
print(f"  Grossier  : {msips_node21_conf.coarse_size}×"
      f"{msips_node21_conf.coarse_size}, "
      f"stride={msips_node21_conf.coarse_stride}")
print(f"  Fin       : {msips_node21_conf.fine_size}×"
      f"{msips_node21_conf.fine_size}, "
      f"stride={msips_node21_conf.fine_stride}")
print(f"  M={msips_node21_conf.M}, I={msips_node21_conf.I}, "
      f"D={msips_node21_conf.D}")

Config MS-IPS NODE21 chargée
  Grossier  : 128×128, stride=128
  Fin       : 64×64, stride=64
  M=10, I=10, D=512


## Cellule 7 — Runner MS-IPS

In [9]:
# ============================================================
# Section 7 — Training loop MS-IPS
# Adapté de iterative.py pour le cas B_seq=1 image entière
# ============================================================
import torch
import numpy as np
from torch import nn
from torch.utils.data import DataLoader

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

conf = msips_node21_conf
torch.manual_seed(conf.seed)
np.random.seed(conf.seed)

# Datasets
train_data = NODE21DatasetMSIPS(conf, train=True)
test_data  = NODE21DatasetMSIPS(conf, train=False)

train_loader = DataLoader(
    train_data, batch_size=1, shuffle=True,
    num_workers=conf.n_worker, pin_memory=conf.pin_memory,
    persistent_workers=True)
test_loader = DataLoader(
    test_data, batch_size=1, shuffle=False,
    num_workers=conf.n_worker, pin_memory=conf.pin_memory,
    persistent_workers=True)

# Modèle
net = MSIPSNet(device, conf).to(device)

# Loss et optimizer
loss_nll = nn.NLLLoss()
loss_bce = nn.BCELoss()
criterions = {}
for task in conf.tasks.values():
    criterions[task['name']] = \
        loss_nll if task['act_fn'] == 'softmax' else loss_bce

optimizer = torch.optim.AdamW(
    net.parameters(), lr=0, weight_decay=conf.wd)

log_train = Logger(conf.tasks)
log_test  = Logger(conf.tasks)

def train_one_epoch_msips(net, criterions, loader,
                           optimizer, device, epoch,
                           log_writer, conf):
    net.train()
    n_prep, start_new_batch = 0, True

    # Placeholders batch
    mem_patches_buf = None
    mem_scales_buf  = None
    labels_buf      = {}
    for task in conf.tasks.values():
        labels_buf[task['name']] = torch.zeros(
            conf.B, dtype=torch.int64).to(device)

    for data_it, data in enumerate(
            loader, start=epoch * len(loader)):

        img    = data['input'].squeeze(0).to(device)  # (1, H, W)
        labels = {t['name']: data[t['name']].to(device)
                  for t in conf.tasks.values()}

        # IPS — sélectionner M patches
        mem_patches, mem_scales, _ = net.ips(img)
        # mem_patches : (M, 1, 256, 256)
        # mem_scales  : (M,)

        if start_new_batch:
            M, C = mem_patches.shape[0], mem_patches.shape[1]
            H_p  = mem_patches.shape[2]
            mem_patches_buf = torch.zeros(
                conf.B, M, C, H_p, H_p).to(device)
            mem_scales_buf  = torch.zeros(
                conf.B, M, dtype=torch.long).to(device)
            start_new_batch = False

        # Remplir le buffer
        mem_patches_buf[n_prep] = mem_patches
        mem_scales_buf[n_prep]  = mem_scales
        for task in conf.tasks.values():
            labels_buf[task['name']][n_prep] = \
                labels[task['name']]
        n_prep += 1

        batch_full   = (n_prep == conf.B)
        is_last_batch= (data_it + 1 == epoch * len(loader)
                        + len(loader))

        if batch_full or is_last_batch:
            if not batch_full:
                mem_patches_buf = mem_patches_buf[:n_prep]
                mem_scales_buf  = mem_scales_buf[:n_prep]
                for task in conf.tasks.values():
                    labels_buf[task['name']] = \
                        labels_buf[task['name']][:n_prep]

            adjust_learning_rate(
                conf.n_epoch_warmup, conf.n_epoch,
                conf.lr, optimizer, loader, data_it + 1)
            optimizer.zero_grad()

            # Forward — traiter chaque image du batch
            B_cur = mem_patches_buf.shape[0]
            loss  = 0
            task_losses, task_preds, task_labels = {}, {}, {}
            for t in conf.tasks.values():
                task_losses[t['name']] = 0
                task_preds[t['name']]  = []
                task_labels[t['name']]  = []

            for b in range(B_cur):
                p   = mem_patches_buf[b]   # (M, C, H, W)
                s   = mem_scales_buf[b]    # (M,)
                preds_b = net(p, s)

                for task in conf.tasks.values():
                    t_name = task['name']
                    pred   = preds_b[t_name].squeeze(-1)
                    label  = labels_buf[t_name][b:b+1]

                    if task['act_fn'] == 'softmax':
                        pred_loss = torch.log(
                            pred + conf.eps)
                        t_loss = criterions[t_name](
                            pred_loss, label)
                    else:
                        t_loss = criterions[t_name](
                            pred.view(-1),
                            label.float().view(-1))

                    task_losses[t_name] += t_loss.item()
                    task_preds[t_name].append(
                        pred.detach().cpu().numpy())
                    task_labels[t_name].append(
                        label.cpu().numpy())
                    loss += t_loss

            loss /= (B_cur * len(conf.tasks))
            loss.backward()
            optimizer.step()

            # Agréger pour le logger
            for task in conf.tasks.values():
                t = task['name']
                task_losses[t] /= B_cur
                task_preds[t]   = np.concatenate(
                    task_preds[t], axis=0)
                task_labels[t]  = np.concatenate(
                    task_labels[t], axis=0)
            log_writer.update(
                task_losses, task_preds, task_labels)

            n_prep = 0
            start_new_batch = True


@torch.no_grad()
def evaluate_msips(net, criterions, loader,
                   device, log_writer, conf):
    net.eval()
    for data in loader:
        img    = data['input'].squeeze(0).to(device)
        labels = {t['name']: data[t['name']].to(device)
                  for t in conf.tasks.values()}

        mem_patches, mem_scales, _ = net.ips(img)
        preds = net(mem_patches, mem_scales)

        task_losses, task_preds, task_labels = {}, {}, {}
        for task in conf.tasks.values():
            t_name = task['name']
            pred   = preds[t_name].squeeze(-1)
            label  = labels[t_name]

            if task['act_fn'] == 'softmax':
                t_loss = criterions[t_name](
                    torch.log(pred + conf.eps), label)
            else:
                t_loss = criterions[t_name](
                    pred.view(-1), label.float().view(-1))

            task_losses[t_name] = t_loss.item()
            task_preds[t_name]  = pred.detach().cpu().numpy()
            task_labels[t_name] = label.cpu().numpy()

        log_writer.update(
            task_losses, task_preds, task_labels)


# ── Entraînement ────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"MS-IPS training sur NODE21")
print(f"Params : {sum(p.numel() for p in net.parameters())/1e6:.2f}M")
print(f"{'='*50}\n")

for epoch in range(conf.n_epoch):

    train_one_epoch_msips(
        net, criterions, train_loader,
        optimizer, device, epoch, log_train, conf)
    log_train.compute_metric()
    log_train.print_stats(
        epoch, train=True,
        lr=optimizer.param_groups[0]['lr'])

    evaluate_msips(
        net, criterions, test_loader,
        device, log_test, conf)
    log_test.compute_metric()
    log_test.print_stats(epoch, train=False)

Train : 3902 images (pos=915, neg=2987)
Test : 980 images (pos=219, neg=761)
MultiScalePatchifier :
  Grossier : 128×128, stride=128 → 64 patches
  Fin      : 64×64, stride=64 → 256 patches
  Total N  : 320 patches

MS-IPS training sur NODE21
Params : 7.82M

Train Epoch: 1
task: nodule, loss: 0.61824, accuracy: 0.71937, auc: 0.48404, f1: 0.13439, precision: 0.24286, recall: 0.09290
avg loss: 0.61824, lr: 1.9999999999999998e-05

Test Epoch: 1
task: nodule, loss: 0.53918, accuracy: 0.77653, auc: 0.55927, f1: 0.00000, precision: 0.00000, recall: 0.00000
avg loss: 0.53918

Train Epoch: 2
task: nodule, loss: 0.56000, accuracy: 0.76217, auc: 0.52233, f1: 0.03132, precision: 0.34884, recall: 0.01639
avg loss: 0.56000, lr: 3.9999999999999996e-05

Test Epoch: 2
task: nodule, loss: 0.55015, accuracy: 0.77653, auc: 0.58659, f1: 0.00000, precision: 0.00000, recall: 0.00000
avg loss: 0.55015

Train Epoch: 3
task: nodule, loss: 0.30774, accuracy: 0.87186, auc: 0.89287, f1: 0.65940, precision: 0.8752